In [ ]:
# ============================================================
# Model Architecture ONLY: Multi-scale CNN + Spectral (FFT) +
# SE block + BiLSTM-Transformer context encoder
#
# NOTE ON NOVELTY: This architecture (multi-scale conv + FFT
# spectral branch + Squeeze-Excitation) closely follows
# MultiScaleSleepNet (Liu et al., 2025, Sensors,
# DOI: 10.3390/s25206328). It is NOT a novel architecture by
# itself -- multi-scale CNN, SE blocks, and spectral fusion all
# individually predate this work. This file only defines the
# model backbone; no loss function is included here.
#
# The intended NEXT step is to attach:
#   (1) artifact-aware per-epoch loss weighting, and
#   (2) SupCon contrastive loss (via a projection head)
# on top of this backbone -- that combination, applied to
# wearable EEG, is the part that has not been previously done
# and would constitute the actual contribution.
#
# This file exports:
#   - SEBlock
#   - MultiScaleCNN   (now with proj_dim-ready output if needed)
#   - BiLSTMTransformerBlock
#   - MultiScaleSleepNetBackbone  (full backbone, NO loss inside)
# ============================================================

import torch
import torch.nn as nn
import torch.nn.functional as F

# ============================================================
# CONFIG (import these constants in your training script too,
# so the model and the data pipeline stay consistent)
# ============================================================
CONTEXT  = 7
WINDOW   = 2 * CONTEXT + 1     # 15
D_MODEL  = 128
DROPOUT  = 0.4
N_HEADS  = 4


# ============================================================
# SQUEEZE-AND-EXCITATION BLOCK
# ============================================================
class SEBlock(nn.Module):
    """Adaptive channel recalibration (Hu et al., 2018)."""
    def __init__(self, channels, reduction=8):
        super().__init__()
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Sequential(
            nn.Linear(channels, channels // reduction), nn.ReLU(),
            nn.Linear(channels // reduction, channels), nn.Sigmoid()
        )

    def forward(self, x):
        # x: (B, C, T)
        b, c, _ = x.shape
        s = self.pool(x).view(b, c)
        s = self.fc(s).view(b, c, 1)
        return x * s


# ============================================================
# MULTI-SCALE CNN + FFT SPECTRAL BRANCH
# ============================================================
class MultiScaleCNN(nn.Module):
    """
    Parallel convolutional branches at different kernel scales
    (fast/slow rhythms) + one FFT-magnitude spectral branch,
    concatenated and recalibrated with SE.
    """
    def __init__(self, in_ch=3, d_model=D_MODEL, dropout=DROPOUT):
        super().__init__()
        mid = d_model // 4  # 4 branches: small/med/large/spectral

        def time_branch(kernel):
            return nn.Sequential(
                nn.Conv1d(in_ch, mid, kernel_size=kernel,
                          stride=6, padding=kernel // 2),
                nn.BatchNorm1d(mid), nn.GELU(),
                nn.MaxPool1d(4, 4),
                nn.Conv1d(mid, mid, kernel_size=8, padding=4),
                nn.BatchNorm1d(mid), nn.GELU(),
                nn.MaxPool1d(2, 2),
                nn.Dropout(dropout),
            )

        self.small  = time_branch(25)   # fast rhythms (spindle/alpha)
        self.medium = time_branch(50)
        self.large  = time_branch(100)  # slow rhythms (delta)

        # Spectral branch: operates on |FFT(x)| instead of raw x
        self.spectral = nn.Sequential(
            nn.Conv1d(in_ch, mid, kernel_size=25, stride=6, padding=12),
            nn.BatchNorm1d(mid), nn.GELU(),
            nn.MaxPool1d(4, 4),
            nn.Conv1d(mid, mid, kernel_size=8, padding=4),
            nn.BatchNorm1d(mid), nn.GELU(),
            nn.MaxPool1d(2, 2),
            nn.Dropout(dropout),
        )

        with torch.no_grad():
            dummy = torch.zeros(1, in_ch, 3000)
            L_s = self.small(dummy).shape[2]
            L_m = self.medium(dummy).shape[2]
            L_l = self.large(dummy).shape[2]
            L_f = self.spectral(dummy).shape[2]

        target_L = min(L_s, L_m, L_l, L_f)
        self.pool_s = nn.AdaptiveAvgPool1d(target_L)
        self.pool_m = nn.AdaptiveAvgPool1d(target_L)
        self.pool_l = nn.AdaptiveAvgPool1d(target_L)
        self.pool_f = nn.AdaptiveAvgPool1d(target_L)
        self.out_len = target_L

        self.se = SEBlock(4 * mid)
        self.proj = nn.Sequential(
            nn.Conv1d(4 * mid, d_model, kernel_size=1),
            nn.BatchNorm1d(d_model), nn.GELU(),
        )

    def forward(self, x):
        # x: (B, C, T) raw time-domain signal
        fs = self.pool_s(self.small(x))
        fm = self.pool_m(self.medium(x))
        fl = self.pool_l(self.large(x))

        # spectral branch: magnitude of real FFT along time axis
        x_fft = torch.fft.rfft(x, dim=-1)
        x_mag = torch.abs(x_fft)
        if x_mag.shape[-1] < x.shape[-1]:
            pad = x.shape[-1] - x_mag.shape[-1]
            x_mag = F.pad(x_mag, (0, pad))
        else:
            x_mag = x_mag[..., :x.shape[-1]]
        ff = self.pool_f(self.spectral(x_mag))

        feat = torch.cat([fs, fm, fl, ff], dim=1)  # (B, 4*mid, L)
        feat = self.se(feat)
        return self.proj(feat)                     # (B, d_model, L)


# ============================================================
# BiLSTM + TRANSFORMER ENCODER BLOCK
# ============================================================
class BiLSTMTransformerBlock(nn.Module):
    def __init__(self, d_model=D_MODEL, n_heads=N_HEADS, dropout=DROPOUT):
        super().__init__()
        self.bilstm = nn.LSTM(
            d_model, d_model // 2, num_layers=1,
            batch_first=True, bidirectional=True
        )
        self.norm1 = nn.LayerNorm(d_model)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads,
            dim_feedforward=d_model * 2, dropout=dropout,
            batch_first=True, activation='gelu'
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=1)
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x):
        # x: (B, L, d_model)
        lstm_out, _ = self.bilstm(x)
        x = self.norm1(x + lstm_out)
        attn_out = self.transformer(x)
        return self.norm2(x + attn_out)


# ============================================================
# FULL BACKBONE (NO LOSS INSIDE)
#
# Returns: logits, center_embedding
#   - logits           : (B, n_classes)  -> for classification loss
#   - center_embedding : (B, d_model)     -> attach a SupCon
#                        projection head to this in your next
#                        training script; also multiply the
#                        classification loss per-sample by your
#                        artifact quality weight there.
# ============================================================
class MultiScaleSleepNetBackbone(nn.Module):
    def __init__(
        self, in_ch=3, d_model=D_MODEL, n_layers=2,
        dropout=DROPOUT, n_classes=5, context=CONTEXT
    ):
        super().__init__()
        self.context = context
        self.d_model = d_model

        self.cnn = MultiScaleCNN(in_ch=in_ch, d_model=d_model, dropout=dropout)
        self.intra_seq_len = self.cnn.out_len

        self.intra_blocks = nn.Sequential(*[
            BiLSTMTransformerBlock(d_model, dropout=dropout)
            for _ in range(n_layers)
        ])

        self.inter_pos = nn.Parameter(torch.randn(1, WINDOW, d_model) * 0.01)
        self.inter_blocks = nn.Sequential(*[
            BiLSTMTransformerBlock(d_model, dropout=dropout)
            for _ in range(n_layers)
        ])

        self.classifier = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, 64), nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(64, n_classes),
        )
        # NOTE: no SupCon projection head here on purpose --
        # add it in your next script as:
        #   self.projector = nn.Sequential(
        #       nn.LayerNorm(d_model),
        #       nn.Linear(d_model, d_model), nn.GELU(),
        #       nn.Linear(d_model, proj_dim),
        #   )
        # and call F.normalize(self.projector(center), dim=1)

    def forward(self, x):
        # x: (B, W, C, T)
        B, W, C, T = x.shape
        cnn_out = self.cnn(x.view(B * W, C, T))          # (B*W, d_model, L)
        cnn_out = cnn_out.permute(0, 2, 1)                # (B*W, L, d_model)
        intra = self.intra_blocks(cnn_out)
        epoch_feat = intra.mean(dim=1).view(B, W, self.d_model)

        inter = epoch_feat + self.inter_pos
        inter = self.inter_blocks(inter)
        center = inter[:, self.context, :]                # (B, d_model)

        logits = self.classifier(center)
        return logits, center


if __name__ == "__main__":
    # quick shape sanity check
    model = MultiScaleSleepNetBackbone(in_ch=3)
    dummy_x = torch.randn(4, WINDOW, 3, 3000)  # (B, W, C, T)
    logits, center = model(dummy_x)
    print("logits:", logits.shape)   # (4, 5)
    print("center:", center.shape)   # (4, 128)
    n_params = sum(p.numel() for p in model.parameters())
    print(f"Parameters: {n_params:,}")

In [2]:
import mne

edf_path = r"D:\22\thesis 3233\Wearanize+_PlugNPlay_v1.0\Wearanize+_PlugNPlay_v1.0\sub-001\eeg\sub-001_task-sleep_eeg.edf"

raw = mne.io.read_raw_edf(edf_path, preload=False, verbose=False)

print(raw)
print(raw.info)

<RawEDF | sub-001_task-sleep_eeg.edf, 39 x 5222400 (20400.0 s), ~41 KiB, data not loaded>
<Info | 8 non-empty values
 bads: []
 ch_names: Zmax_EEGL, Zmax_EEGR, Zmax_ACCX, Zmax_ACCY, Zmax_ACCZ, ...
 chs: 39 EEG
 custom_ref_applied: False
 highpass: 0.0 Hz
 lowpass: 128.0 Hz
 meas_date: 2000-01-01 00:00:00 UTC
 nchan: 39
 projs: []
 sfreq: 256.0 Hz
 subject_info: <subject_info | his_id: X, sex: 0, last_name: Sub001s1>
>


In [4]:
import os

folder = r"D:\22\thesis 3233\Wearanize+_PlugNPlay_v1.0\Wearanize+_PlugNPlay_v1.0\sub-001\eeg"

print("Folder exists:", os.path.exists(folder))
print("\nFiles:\n")

for f in os.listdir(folder):
    print(f)

Folder exists: True

Files:

sub-001_task-sleep_channels.json
sub-001_task-sleep_channels.tsv
sub-001_task-sleep_eeg.edf
sub-001_task-sleep_eeg.json


In [11]:
import pandas as pd

tsv = r"D:\22\thesis 3233\Wearanize+_PlugNPlay_v1.0\Wearanize+_PlugNPlay_v1.0\sub-007\eeg\sub-007_task-sleep_channels.tsv"

df = pd.read_csv(tsv, sep="\t")

print(df.columns.tolist())
print("\n")
print(df.to_string(index=False))

['name', 'type', 'units', 'sampling_frequency', 'max_val', 'min_val', 'duration', 'description']


            name  type    units  sampling_frequency     max_val      min_val  duration                         description
       Zmax_EEGL   EEG       µV          256.000000 1975.939000 -1976.000000   32340.0            Device: Zmax (Hypnodyne)
       Zmax_EEGR   EEG       µV          256.000000 1975.939000 -1976.000000   32340.0            Device: Zmax (Hypnodyne)
       Zmax_ACCX ACCEL        g          256.000000    1.659179    -1.476560   32340.0            Device: Zmax (Hypnodyne)
       Zmax_ACCY ACCEL        g          256.000000    1.999023    -1.936520   32340.0            Device: Zmax (Hypnodyne)
       Zmax_ACCZ ACCEL        g          256.000000    1.985351    -1.044920   32340.0            Device: Zmax (Hypnodyne)
      Zmax_NOISE  MISC Unitless          256.000000    0.001000     0.000000   32340.0            Device: Zmax (Hypnodyne)
  Zmax_OXY_IR_AC   PPG Unitless         

In [14]:
import mne

edf = r"D:\22\thesis 3233\Wearanize+_PlugNPlay_v1.0\Wearanize+_PlugNPlay_v1.0\sub-007\eeg\sub-007_task-sleep_eeg.edf"

raw = mne.io.read_raw_edf(edf, preload=False)

print(raw.info["sfreq"])
print("Emp_BVP" in raw.ch_names)
print("Emp_HR" in raw.ch_names)
print("Emp_TEMP" in raw.ch_names)
print(raw.get_data(picks=["Emp_BVP"]).shape)
print(raw.get_data(picks=["Emp_HR"]).shape)
print(raw.get_data(picks=["Emp_TEMP"]).shape)

Extracting EDF parameters from D:\22\thesis 3233\Wearanize+_PlugNPlay_v1.0\Wearanize+_PlugNPlay_v1.0\sub-007\eeg\sub-007_task-sleep_eeg.edf...
Setting channel info structure...
Creating raw.info structure...
256.0
True
True
True
(1, 8279040)
(1, 8279040)
(1, 8279040)


In [8]:
import pandas as pd

tsv = r"D:\22\thesis 3233\Wearanize+_PlugNPlay_v1.0\Wearanize+_PlugNPlay_v1.0\sub-001\eeg\sub-001_task-sleep_channels.tsv"

df = pd.read_csv(tsv, sep="\t")

print(df.columns.tolist())
print("\n")
print(df.to_string(index=False))

['name', 'type', 'units', 'sampling_frequency', 'max_val', 'min_val', 'duration', 'description']


            name  type    units  sampling_frequency     max_val     min_val  duration                         description
       Zmax_EEGL   EEG       µV          256.000000 1975.939000 -1976.00000   20400.0            Device: Zmax (Hypnodyne)
       Zmax_EEGR   EEG       µV          256.000000 1975.939000 -1976.00000   20400.0            Device: Zmax (Hypnodyne)
       Zmax_ACCX ACCEL        g          256.000000    1.999023    -2.00000   20400.0            Device: Zmax (Hypnodyne)
       Zmax_ACCY ACCEL        g          256.000000    1.999023    -1.97851   20400.0            Device: Zmax (Hypnodyne)
       Zmax_ACCZ ACCEL        g          256.000000    1.999023    -0.60253   20400.0            Device: Zmax (Hypnodyne)
      Zmax_NOISE  MISC Unitless          256.000000    0.001000     0.00000   20400.0            Device: Zmax (Hypnodyne)
  Zmax_OXY_IR_AC   PPG Unitless          256.00